# Grokking Reproduction — Nanda et al. (2023)
**Paper:** Progress Measures for Grokking via Mechanistic Interpretability  
**DOI:** 10.48550/arXiv.2301.05217  

Run cells in order. Each cell is independent and re-runnable after a `git pull`.

In [ ]:
import os, sys
from pathlib import Path

REPO_URL = "https://github.com/MrFaruk0/grokking-reproduction.git"
REPO_DIR = Path("/content/grokking-reproduction")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}
sys.path.insert(0, str(REPO_DIR))
!pip install einops plotly pandas -q
Path("outputs").mkdir(exist_ok=True)
print("Setup complete.")

In [ ]:
print(Path("PROJECT_MEMORY.md").read_text())

In [ ]:
import torch
from transformers import Config, train_model, full_loss, gen_train_test

config = Config(
    p=113, d_model=128, num_heads=4, d_mlp=512,
    num_layers=1, frac_train=0.3, num_epochs=50000,
    seed=0, act_type='ReLU', use_ln=False, fn_name='add',
)
world = train_model(config)

# Training bittikten sonra accuracy hesapla
all_data = torch.tensor(
    [(i, j, 113) for i in range(113) for j in range(113)]
).to(config.device)

with torch.no_grad():
    logits = world.model(all_data)[:, -1, :-1]
    labels = torch.tensor([config.fn(i,j) for i,j,_ in all_data]).to(config.device)
    preds = logits.argmax(-1)
    
    train_data, test_data = gen_train_test(config)
    is_train, is_test = config.is_train_is_test(train_data)
    
    train_acc = (preds[is_train] == labels[is_train]).float().mean().item()
    test_acc  = (preds[is_test]  == labels[is_test]).float().mean().item()

print(f"Final train accuracy: {train_acc:.4f}")
print(f"Final test accuracy:  {test_acc:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

fig, ax = plt.subplots(figsize=(10, 4))
epochs = list(range(len(world.train_losses)))
ax.plot(epochs, world.train_losses, label='Train loss')
ax.plot(epochs, world.test_losses,  label='Test loss')
ax.set_yscale('log')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss (log)')
ax.set_title('Figure 1: Grokking — Train vs Test Loss')
ax.legend()
fig.tight_layout()
fig.savefig('outputs/figure1_grokking_curve.png', dpi=150)
plt.show()

In [ ]:
from figures import plot_fourier_spectrum
plot_fourier_spectrum(world.model, config, save_dir='outputs')

In [ ]:
from explorations import sweep_weight_decay
from figures import plot_weight_decay_sweep

wd_results = sweep_weight_decay(config, weight_decays=[0.0, 0.1, 0.5, 1.0, 2.0])
plot_weight_decay_sweep(wd_results, save_dir='outputs')

In [ ]:
from explorations import sweep_prime_p
from figures import plot_p_sweep

p_results = sweep_prime_p(config, primes=[53, 97, 113, 127])
plot_p_sweep(p_results, save_dir='outputs')

In [ ]:
from explorations import sweep_operations
from figures import plot_operations_sweep

op_results = sweep_operations(config, operations=['add', 'subtract', 'multiply'])
plot_operations_sweep(op_results, save_dir='outputs')

In [ ]:
from explorations import sweep_depth
from figures import plot_depth_sweep

depth_results = sweep_depth(config, depths=[1, 2, 3])
plot_depth_sweep(depth_results, save_dir='outputs')